In [1]:
def plot_figures(outputs,directory=''):
    
    global gdf
    
    gdf = copy.deepcopy(neighborhoods)
    gdf.fillna(0)
    
    if 'Text Output' in outputs:
            text_res = write_text_results()
            print(text_res)
    
    elif 'Loss breakdown' in outputs:
        map_figure = plot_loss_breakdown()
        
    elif 'Investment breakdown' in outputs:
        map_figure = plot_investment_breakdown()
    
    elif 'Download data (.shp)' in outputs:
        gdf.to_file(rf"{directory}Outputs/output_data.shp")
        print('Data downloaded to backend/Data/Outputs/output_data')
        
    else:
        i = np.where(np.array(options[1:-3])==outputs)[0][0]
        labels = np.array(['Residential structural retrofit only (%)','Residential shelter only (%)',
                          'Residential shelter and retrofit (%)', 
                          'Any structural retrofit (%) for all buildings','Benefit-cost ratio',
                          'Percent minority  (%)','Percent renter (%)','Population'])
        columns = np.array(['retrofit','shelter','both','retro_any','BCR','%Minority',
                           'PercRenter','POP20'])
        vmax = np.array([100,100,100,100,None,None,None,None])
        
        if vmax[i] != None:
            map_figure = plot_map_output(columns[i],label=labels[i],vmax=vmax[i])
        else:
            map_figure = plot_map_output(columns[i],label=labels[i])

In [2]:
def plot_investment_breakdown():
    # Plot power investments
    fig, ax = plt.subplots(figsize=(8,5))

    plt.bar(np.arange(perc_retro.shape[0]),perc_retro,linewidth=np.where(perc_retro == 0, 0, 2),edgecolor='black')
    plt.ylim([-0.01,1.01])
    plt.xlabel('Retrofit option for residential buildings')
    plt.ylabel('Proportion of strategy application')
    labels = ['','No action', 'R1', 'R2', 'Shelter', 'R1 + Shelter', 'R2 + Shelter']
    ax.set_xticklabels(labels) 

In [ ]:
def plot_loss_breakdown(ax=None):
    
    if ax == None:
        fig, ax = plt.subplots()
    total_debris_cost = debris_cost_bld
    total_debris_cost_0 =  debris_cost_bld_0
   
    outer_sizes = np.array([bld_cost_0.sum(),dislocation_cost_0.sum(),casualty_cost_0.sum(),total_debris_cost_0.sum()])
    inner_sizes = np.array([bld_cost_0.sum()-bld_cost.sum(),bld_cost.sum(),
                            dislocation_cost_0.sum()-dislocation_cost.sum(),dislocation_cost.sum(),
                            casualty_cost_0.sum()-casualty_cost.sum(),casualty_cost.sum(),
                            total_debris_cost_0.sum()-total_debris_cost.sum(),total_debris_cost.sum(),])/outer_sizes.sum()
    labels = np.array(['Building repair', 'Dislocation', 'Casualties','Debris removal',])
    outer_colors = ['#499894', '#B07AA1', '#F28E2B', '#59A14F', '#4E79A7']
    inner_colors = ['white', '#86BCB6', 'white', '#D4A6C8', 'white', '#FFBE7D',
                    'white', '#8CD17D', 'white', '#A0CBE8']

    plt.pie(outer_sizes,colors=outer_colors, startangle=90,frame=True, radius=4,autopct='%1.0f%%'
            ,textprops={'size':12}, pctdistance=0.9)
    plt.pie(inner_sizes,colors=inner_colors,radius=3,startangle=90)
    center_circle = plt.Circle((0,0), 2, color='black', fc='white', linewidth=0)
    fig = plt.gcf()
    fig.gca().add_artist(center_circle)
    fig.set_size_inches(6,6)
    plt.axis('equal')
    plt.tight_layout()
    plt.legend(labels,loc='center')
    plt.text(-1.52,-4.2,'Outer circle: Original loss breakdown')
    plt.text(-1.6,-4.4, 'Inner circle: Retrofitted loss breakdown')
    plt.show()

In [1]:
def plot_map_output(col,label,vmin=0,vmax=None,categorical=False):
    fig, ax = plt.subplots()
    # Create colormap
    
    plt.rcParams["font.family"] = "Times New Roman"
    
    #gdf[gdf[col]>0].plot(ax=ax,column=col,legend=True,edgecolor='gray',linewidth=0.15,cmap='jet' ,vmin=vmin,vmax=vmax,legend_kwds={'label':label})
    gdf.plot(ax=ax,column=col,legend=True,edgecolor='gray',linewidth=0.15,cmap='jet' ,vmin=vmin,vmax=vmax,legend_kwds={'label':label})

    #gdf[gdf[col]>0].plot(ax=ax,column=col,legend=True,edgecolor='gray',linewidth=0.15,cmap='jet' ,vmin=vmin,vmax=vmax,legend_kwds={'label':label})
    #gdf[gdf[col]<=0].plot(ax=ax,column=col,legend=True,edgecolor='gray',linewidth=0.15,color='white')

    ax.tick_params(left=False,right=False,labelleft=False,labelbottom=False,bottom=False)

    cx.add_basemap(ax=ax,alpha=0.5,crs=neighborhoods.crs,attribution='')

    add_scalebar(ax)

In [ ]:
def write_text_results():
    
    text_res = []
    text_res.append('Maximum retrofit spending is $' + str(round(total_spending,2)) + ' million')
    text_res.append('Shelter ' + str(round(sheltered_homes)) + ' (' + str(round(res_shelt_perc,1)) + ' %) homes')
    text_res.append('Upgrade ' + str(round(residential_retrofit_1)) + ' (' + str(round(res_1_perc,1)) + ' %) residential homes by Strategy 1')
    text_res.append('Upgrade ' + str(round(residential_retrofit_2)) + ' (' + str(round(res_2_perc,1)) +' %) residential homes by Strategy 2')
    text_res.append('Upgrade ' + str(round(nonres_retrofit_1)) + ' (' + str(round(nonres_1_perc,1)) +' %) nonresidential homes by Strategy 1')
    text_res.append('Upgrade ' + str(round(nonres_retrofit_2)) + ' (' + str(round(nonres_2_perc,1)) + ' %) nonresidential homes by Strategy 2')
    
    text_rest = str(text_res)
    
    return str("\n".join(text_res))

In [ ]:
def add_scalebar(ax):
    scalebar_length = 2000 # length, m
    scalebar = AnchoredSizeBar(ax.transData,
                               scalebar_length,          # length of the bar in data units
                                f'{int(scalebar_length/1000)} km',   # label text
                               loc='lower right',        # location of the bar
                               pad=0.1,                  # padding between the bar and the label
                               color='black',            # color of the bar and label
                               frameon=False,            # if True, draw a box around the scalebar
                               size_vertical=1,          # thickness of the scalebar
                               fontproperties=fm.FontProperties(size=10)) # font properties for the label


    ax.add_artist(scalebar)